# **Quick notebook on Foundation of NLP**

In [ ]:
%%capture
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install --upgrade pythainlp
!pip install datasets

## **English Word-level tokenization**

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, token.pos_, token.dep_)

In [ ]:
for token in doc:
    print(token.text)

## **Thai word-level tokenization**

- PyThaiNLP provides various tokenization engine for word-level tokenization

In [ ]:
from pythainlp.tokenize import word_tokenize

In [ ]:
text = "โอเคบ่พวกเรารักภาษาบ้านเกิด"
word_tokenize(text, engine="newmm")

In [ ]:
text = "โอเคบ่พวกเรารักภาษาบ้านเกิด"
word_tokenize(text, engine="deepcut")

## **Bag-of-word for Text Classification**

In [ ]:
import pandas as pd
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import precision_recall_fscore_support
from datasets import load_dataset

In [ ]:
dataset = load_dataset("prachathai67k", split="train")
classes = [
  'politics', 'human_rights', 'quality_of_life', 'international', 'social',
  'environment', 'economics', 'culture', 'labor', 'national_security', 'ict',
  'education'
]
texts = [r["body_text"] for r in dataset]
labels = [{k: v for k, v in r.items() if k in classes} for r in dataset]
labels_df = pd.DataFrame(labels)

In [ ]:
# transform a few text examples
tfidf = TfidfVectorizer(
    sublinear_tf=True, min_df=5,
    ngram_range=(1, 1),
    tokenizer=lambda x: word_tokenize(x, engine="newmm")  # only specify for Thai language
)

In [ ]:
# for testing, we will only use 5000 datapoints for training the model
X_tfidf = tfidf.fit_transform(texts[0:5000])
y = labels_df.values[0: 5000, :]

In [ ]:
logist_model = MultiOutputClassifier(LogisticRegression())
logist_model.fit(X_tfidf, y)
y_pred = logist_model.predict(X_tfidf)

In [ ]:
metrics_list = []

for i, col_name in enumerate(labels_df.columns):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y[:, i], y_pred[:, i], average="weighted"
    )
    metrics_list.append({
        "News Type": col_name,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

results_df = pd.DataFrame(metrics_list)
results_df = results_df.set_index("News Type").round(3)
print(results_df.sort_values(by="F1-Score", ascending=False))

## **Topic Modeling: LSA, LDA**

In [ ]:
# import library
import io
import urllib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from pythainlp import word_tokenize

font_url = "https://github.com/Phonbopit/sarabun-webfont/raw/refs/heads/master/fonts/thsarabunnew-webfont.ttf"
response = urllib.request.urlopen(font_url)
font_data = io.BytesIO(response.read())

with open("thsarabunnew-webfont.ttf", "wb") as f:
    f.write(font_data.getbuffer())

fm.fontManager.addfont("thsarabunnew-webfont.ttf")
plt.rcParams['font.family'] = 'TH Sarabun New'

In [ ]:
count_vec = CountVectorizer(
    min_df=3, max_df=0.9,
    ngram_range=(1, 1),
    tokenizer=lambda x: word_tokenize(x, engine="newmm")  # only specify for Thai language
)
tfidf_vec = TfidfVectorizer(
    min_df=3, max_df=0.9,
    ngram_range=(1, 1),
    tokenizer=lambda x: word_tokenize(x, engine="newmm")  # only specify for Thai language
)

X_count = count_vec.fit_transform(texts[0:5000])
X = tfidf_vec.fit_transform(texts[0:5000])

In [ ]:
def plot_top_words(model, feature_names, n_top_words: int = 20, title: str = ""):
    n_topics = len(model.components_)
    n_cols = 5
    n_rows = math.ceil(n_topics / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(30, 7 * n_rows), sharex=True)
    axes = axes.flatten()
    
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f'Topic {topic_idx + 1}', fontdict={'fontsize': 30})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=20)
        
        for i in 'top right left'.split():
            ax.spines[i].set_visible(False)
            
    for i in range(topic_idx + 1, len(axes)):
        axes[i].axis('off')

    fig.suptitle(title, fontsize=40)
    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()

In [ ]:
# get feature names
tfidf_feature_names = tfidf_vec.get_feature_names_out()
count_feature_names = count_vec.get_feature_names_out()

In [ ]:
# fitting NMF
nmf = NMF(n_components=20, random_state=1).fit(X)

In [ ]:
# topics using NMF
plot_top_words(nmf, tfidf_feature_names, n_top_words=20,
               title='Topics in NMF model')

In [ ]:
# fitting LDA model, takes some time
lda = LatentDirichletAllocation(n_components=20, random_state=0).fit(X_count)

In [ ]:
# topics using LDA
plot_top_words(lda, count_feature_names, n_top_words=20,
               title='Topics in LDA model')